# 08 — Surrogate Model + Optimisation: Inverse Design of Underwater Acoustic Coatings

## Overview
This notebook implements a **two-phase surrogate-based optimisation** approach:

1. **Phase 1** — Train a fast forward surrogate (Gradient Boosting, small NN, and Gaussian Process) that maps 20 design parameters → average absorption.
2. **Phase 2** — For a given target absorption, use black-box optimisation to find parameters that minimise `|surrogate(params) - target|²`.

Three optimisers are compared:
- **Differential Evolution** (scipy)
- **Genetic Algorithm** (pymoo)
- **Particle Swarm Optimisation** (custom implementation)

No deep learning is required — this is the most classical engineering approach.

In [ ]:
# Install pymoo if needed
%pip install pymoo -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from scipy.optimize import differential_evolution

from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.optimize import minimize as pymoo_minimize
from pymoo.core.problem import ElementwiseProblem
from pymoo.termination import get_termination

warnings.filterwarnings('ignore')
np.random.seed(42)

## TMM Forward Model (pure NumPy)

In [ ]:
RHO_WATER = 1000.0
C_WATER   = 1500.0
Z_WATER   = RHO_WATER * C_WATER
W_FILM    = 2.0
HOLLOW_MAP = {1: 0, 2: 1, 4: 2, 5: 3, 7: 4, 8: 5}

PARAM_COLUMNS = (
    [f'd{i}' for i in range(1, 11)] +
    ['m2', 'm3', 'm5', 'm6', 'm8', 'm9'] +
    ['rho', 'eta', 'E', 'nu']
)

PARAM_BOUNDS = np.array([
    [1.0, 20.0], [1.0, 20.0], [1.0, 20.0], [1.0, 20.0], [1.0, 20.0],
    [1.0, 20.0], [1.0, 20.0], [1.0, 20.0], [1.0, 20.0], [1.0, 20.0],
    [20.0, 1980.0], [20.0, 1980.0], [20.0, 1980.0],
    [20.0, 1980.0], [20.0, 1980.0], [20.0, 1980.0],
    [1000.0, 1500.0], [0.1, 0.8], [1e7, 1e8], [0.4, 0.49],
])


def tmm_forward(params):
    d_mm = np.asarray(params[:10], dtype=float)
    m_mm = np.asarray(params[10:16], dtype=float)
    rho_r, eta, E_r, nu = float(params[16]), float(params[17]), float(params[18]), float(params[19])
    d_m = d_mm / 1000.0
    m_m = m_mm / 1000.0
    E_complex = E_r * (1.0 + 1j * eta)
    lam = (E_complex * nu) / ((1 + nu) * (1 - 2*nu))
    mu  = E_complex / (2.0 * (1 + nu))
    alpha_sum = 0.0
    for f in range(1, 1001):
        omega = 2.0 * np.pi * f
        T = np.eye(2, dtype=complex)
        for i in range(10):
            d_i = d_m[i]
            eps = m_m[HOLLOW_MAP[i]] / W_FILM if i in HOLLOW_MAP else 0.0
            rho_eff = rho_r * (1.0 - eps**2)
            num = mu*(lam + 2*mu)*(eps**2 + 1) + 2*eps**2*lam
            den = (lam + mu)*eps**2 + mu
            S_eff = num / den if abs(den) > 1e-15 else num * 1e15
            c_eff = np.sqrt(S_eff / rho_eff)
            k = omega / c_eff
            Z_eff = rho_eff * c_eff
            cos_kd = np.cos(k * d_i)
            sin_kd = np.sin(k * d_i)
            t21 = (1j * sin_kd / Z_eff) if abs(Z_eff) > 1e-15 else (1e15 + 0j)
            Ti = np.array([[cos_kd, 1j*Z_eff*sin_kd], [t21, cos_kd]], dtype=complex)
            T = T @ Ti
        if abs(T[1, 0]) < 1e-15:
            R = 1.0
        else:
            Z_in = T[0, 0] / T[1, 0]
            R = (Z_in - Z_WATER) / (Z_in + Z_WATER) if abs(Z_in) < 1e15 else 1.0
        a = 1.0 - abs(R)**2
        alpha_sum += max(0.0, min(1.0, a.real if hasattr(a, 'real') else a))
    return alpha_sum / 1000.0


_base = [10]*10 + [1000]*6 + [1130, 0.4, 5e7, 0.44]
print(f'Base-case absorption: {tmm_forward(_base):.4f}')

## Data Loading & Preprocessing

In [ ]:
df = pd.read_csv('../data/lhs_data.csv')
print(f'Dataset shape: {df.shape}')

param_cols = [c for c in df.columns if c != 'Average_Absorption']
X = df[param_cols].values
y = df['Average_Absorption'].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print(f'Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}')

param_scaler = MinMaxScaler()
X_train_n = param_scaler.fit_transform(X_train)
X_val_n   = param_scaler.transform(X_val)
X_test_n  = param_scaler.transform(X_test)

## Phase 1 — Train & Compare Surrogate Models
We train three forward surrogates and select the best one for the optimisation phase.

In [ ]:
# --- Surrogate A: Gradient Boosting Regressor ---
print('Training Gradient Boosting surrogate ...')
t0 = time.time()
gbr = GradientBoostingRegressor(
    n_estimators=500, max_depth=8, learning_rate=0.05,
    subsample=0.8, random_state=42
)
gbr.fit(X_train_n, y_train)
gbr_time = time.time() - t0
gbr_val_r2 = r2_score(y_val, gbr.predict(X_val_n))
print(f'  GBR  val R²={gbr_val_r2:.4f}  time={gbr_time:.1f}s')

In [ ]:
# --- Surrogate B: MLP Regressor ---
print('Training MLP surrogate ...')
t0 = time.time()
mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32), activation='relu',
    max_iter=300, early_stopping=True, random_state=42
)
mlp.fit(X_train_n, y_train)
mlp_time = time.time() - t0
mlp_val_r2 = r2_score(y_val, mlp.predict(X_val_n))
print(f'  MLP  val R²={mlp_val_r2:.4f}  time={mlp_time:.1f}s')

In [ ]:
# --- Surrogate C: Gaussian Process (subset) ---
print('Training GP surrogate (5k subset) ...')
gp_idx = np.random.choice(len(X_train_n), 5000, replace=False)
t0 = time.time()
kernel = ConstantKernel(1.0) * RBF(length_scale=np.ones(20))
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=2, random_state=42)
gpr.fit(X_train_n[gp_idx], y_train[gp_idx])
gp_time = time.time() - t0
gp_val_r2 = r2_score(y_val, gpr.predict(X_val_n))
print(f'  GP   val R²={gp_val_r2:.4f}  time={gp_time:.1f}s')

In [ ]:
# Select best surrogate
scores = {'GBR': gbr_val_r2, 'MLP': mlp_val_r2, 'GP': gp_val_r2}
models = {'GBR': gbr, 'MLP': mlp, 'GP': gpr}
best_name = max(scores, key=scores.get)
best_surrogate = models[best_name]
print(f'\nBest surrogate: {best_name} (R²={scores[best_name]:.4f})')

def surrogate_predict(params_norm):
    """Predict absorption from normalised (20,) params."""
    return best_surrogate.predict(params_norm.reshape(1, -1))[0]

## Phase 2 — Optimisation

For a given target absorption, find normalised parameters that minimise `(surrogate(x) - target)²`.

In [ ]:
BOUNDS_NORM = [(0.0, 1.0)] * 20  # normalised bounds

# --- Optimiser A: Differential Evolution ---
def run_diffevo(target, seed=42):
    def obj(x):
        return (best_surrogate.predict(x.reshape(1, -1))[0] - target) ** 2
    result = differential_evolution(obj, BOUNDS_NORM, maxiter=300, tol=1e-10,
                                    seed=seed, polish=True)
    return result.x, result.fun


# --- Optimiser B: Genetic Algorithm (pymoo) ---
class AbsorptionProblem(ElementwiseProblem):
    def __init__(self, target):
        super().__init__(n_var=20, n_obj=1, xl=0.0, xu=1.0)
        self.target = target
    def _evaluate(self, x, out, *args, **kwargs):
        pred = best_surrogate.predict(x.reshape(1, -1))[0]
        out['F'] = [(pred - self.target) ** 2]

def run_ga(target, seed=42):
    problem = AbsorptionProblem(target)
    algorithm = GA(pop_size=100)
    termination = get_termination('n_gen', 200)
    res = pymoo_minimize(problem, algorithm, termination, seed=seed, verbose=False)
    return res.X, res.F[0]


# --- Optimiser C: Particle Swarm Optimisation (custom) ---
def run_pso(target, n_particles=50, n_iters=200, seed=42):
    rng = np.random.RandomState(seed)
    dim = 20
    pos = rng.uniform(0, 1, (n_particles, dim))
    vel = rng.uniform(-0.1, 0.1, (n_particles, dim))
    pbest_pos = pos.copy()
    pbest_val = np.full(n_particles, np.inf)
    gbest_pos = None
    gbest_val = np.inf

    w, c1, c2 = 0.7, 1.5, 1.5

    for _ in range(n_iters):
        preds = best_surrogate.predict(pos)
        fitness = (preds - target) ** 2
        improved = fitness < pbest_val
        pbest_val[improved] = fitness[improved]
        pbest_pos[improved] = pos[improved]
        best_idx = np.argmin(pbest_val)
        if pbest_val[best_idx] < gbest_val:
            gbest_val = pbest_val[best_idx]
            gbest_pos = pbest_pos[best_idx].copy()
        r1 = rng.uniform(0, 1, (n_particles, dim))
        r2 = rng.uniform(0, 1, (n_particles, dim))
        vel = w*vel + c1*r1*(pbest_pos - pos) + c2*r2*(gbest_pos - pos)
        pos = np.clip(pos + vel, 0, 1)

    return gbest_pos, gbest_val

In [ ]:
# Run all 3 optimisers on 50 test targets with 3 random restarts each
N_TARGETS = 50
N_RESTARTS = 3
test_targets = y_test[:N_TARGETS]

results_de, results_ga, results_pso = [], [], []
de_params_all, ga_params_all, pso_params_all = [], [], []

print(f'Running optimisation on {N_TARGETS} targets x {N_RESTARTS} restarts ...')
t0 = time.time()

for i, target in enumerate(test_targets):
    best_de, best_ga, best_pso = (None, np.inf), (None, np.inf), (None, np.inf)
    for s in range(N_RESTARTS):
        seed = 42 + s
        x_de, f_de = run_diffevo(target, seed=seed)
        if f_de < best_de[1]: best_de = (x_de, f_de)

        x_ga, f_ga = run_ga(target, seed=seed)
        if f_ga < best_ga[1]: best_ga = (x_ga, f_ga)

        x_pso, f_pso = run_pso(target, seed=seed)
        if f_pso < best_pso[1]: best_pso = (x_pso, f_pso)

    de_params_all.append(best_de[0])
    ga_params_all.append(best_ga[0])
    pso_params_all.append(best_pso[0])
    results_de.append(best_de[1])
    results_ga.append(best_ga[1])
    results_pso.append(best_pso[1])

    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{N_TARGETS} done')

opt_time = time.time() - t0
print(f'Optimisation finished in {opt_time:.1f}s')

## Physics Validation
Denormalise optimised params and run TMM to get the true absorption.

In [ ]:
def validate_with_tmm(params_norm_list, targets):
    params_phys = param_scaler.inverse_transform(np.array(params_norm_list))
    recon = np.array([tmm_forward(p) for p in params_phys])
    mse = mean_squared_error(targets, recon)
    mae = mean_absolute_error(targets, recon)
    r2  = r2_score(targets, recon)
    return recon, mse, mae, r2

recon_de, mse_de, mae_de, r2_de = validate_with_tmm(de_params_all, test_targets)
recon_ga, mse_ga, mae_ga, r2_ga = validate_with_tmm(ga_params_all, test_targets)
recon_pso, mse_pso, mae_pso, r2_pso = validate_with_tmm(pso_params_all, test_targets)

print(f'Diff. Evo.   TMM recon  MSE={mse_de:.6f}  MAE={mae_de:.4f}  R²={r2_de:.4f}')
print(f'GA (pymoo)   TMM recon  MSE={mse_ga:.6f}  MAE={mae_ga:.4f}  R²={r2_ga:.4f}')
print(f'PSO          TMM recon  MSE={mse_pso:.6f}  MAE={mae_pso:.4f}  R²={r2_pso:.4f}')

## Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, recon, name, r2 in [
    (axes[0], recon_de,  'Diff. Evolution', r2_de),
    (axes[1], recon_ga,  'Genetic Algo.',   r2_ga),
    (axes[2], recon_pso, 'PSO',             r2_pso),
]:
    ax.scatter(test_targets, recon, s=15, alpha=0.7)
    ax.plot([0, 0.7], [0, 0.7], 'r--')
    ax.set_xlabel('Target Absorption')
    ax.set_ylabel('TMM Reconstructed')
    ax.set_title(f'{name} (R²={r2:.3f})')

plt.tight_layout()
plt.show()

In [ ]:
# Box plot of reconstruction errors
errors_de  = np.abs(test_targets - recon_de)
errors_ga  = np.abs(test_targets - recon_ga)
errors_pso = np.abs(test_targets - recon_pso)

fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([errors_de, errors_ga, errors_pso],
           labels=['Diff. Evo.', 'GA', 'PSO'])
ax.set_ylabel('|Target - TMM Reconstructed|')
ax.set_title('Reconstruction Error Comparison')
plt.tight_layout()
plt.show()

In [ ]:
# Surrogate comparison bar chart
fig, ax = plt.subplots(figsize=(5, 3))
names = list(scores.keys())
vals  = [scores[n] for n in names]
ax.bar(names, vals, color=['steelblue', 'coral', 'seagreen'])
ax.set_ylabel('Validation R²')
ax.set_title('Surrogate Model Comparison')
ax.set_ylim(min(vals) - 0.05, 1.0)
plt.tight_layout()
plt.show()

## Results Summary

In [ ]:
# Pick best optimiser
opt_results = {'DiffEvo': (mse_de, mae_de, r2_de),
               'GA': (mse_ga, mae_ga, r2_ga),
               'PSO': (mse_pso, mae_pso, r2_pso)}
best_opt = max(opt_results, key=lambda k: opt_results[k][2])
best_mse, best_mae, best_r2 = opt_results[best_opt]

results = {
    'method': 'Surrogate_Optimization',
    'best_surrogate': best_name,
    'surrogate_r2': float(scores[best_name]),
    'best_optimizer': best_opt,
    'recon_mse': float(best_mse),
    'recon_mae': float(best_mae),
    'recon_r2':  float(best_r2),
    'can_generate_diverse': True,
    'total_time_seconds': round(opt_time, 1),
}
print(results)

### Discussion

The surrogate-based optimisation approach is the most **practical and interpretable** method. The surrogate accuracy is the primary bottleneck: any error in the surrogate propagates to the optimiser's solution.

**Advantages**: No neural-network inverse model needed; multiple optimisers can be compared; multi-start gives diverse solutions; the approach is well-understood and debuggable.

**Limitations**: Each optimisation run is independent, so batch inference is slower than a trained inverse NN. The surrogate may not capture edge cases of the parameter space accurately.